# Evaluating Fitbit step estimates against national estimates in Japan (NHNS-J) and England (ELSA Wave 10)

Author: Bram Schonfeldt

This notebook executes a unified, harmonized analytical evaluation of commercial
wearable step counts (Fitbit) against national benchmark studies in Japan
(Japanese National Health and Nutrition Survey; NHNS-J 2015-2023) and England
(English Longitudinal Study of Ageing; ELSA Wave 10, 2021-2023).

The analytical pipeline is organized into three structured sections conforming to
Google Python and Jupyter Style Guidelines:

1. Section 1: Detailed Discrepancy Analysis in Japan (N = 84 Stratum-Year Cells)
   Contrasts Difference of Population Means against Mean of Stratum-Level
   Differences across raw Fitbit step counts, Nakagata free-living calibration
   (All Ages 20-70+), and Nakagata calibration specifically within its target
   demographic bracket (Ages 20-50), accompanied by stratum-clustered bootstrap
   95% confidence intervals.
2. Section 2: Harmonized Cross-Country Pipeline
   Executes symmetrical marginal ordinary least squares (OLS) step modeling
   (Y ~ C(age) + C(gender)) for England (adults aged 50+) and Japan
   (All Ages 20-70+, N = 84 stratum-year cells), propagating analytical
   uncertainty via the first-order Delta Method.
3. Section 3: Harmonized Subgroup and Composite Summary Tables and Figures
   Packages the cross-country marginal evaluations into a publication-ready
   Master Subgroup Accuracy Table, LaTeX table code, and comparative figure.

Note: The longitudinal Japan trajectory and COVID-19 gap-filling analysis
(2020-2021) is conducted in `4_filling_gaps_japan.ipynb`.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import pickle as pkl
import sys
from typing import Any, Dict, List, Tuple

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from patsy.builtins import C as Cat
import statsmodels.formula.api as smf

# Set rcParams
plt.rcdefaults()
plt.rcParams['figure.dpi'] = 200

sys.path.insert(0, os.path.abspath('../..'))
import stratify as strat

Load in the data...

In [ ]:
internal_dir = strat.get_internal_data_dir()
public_dir = strat.get_public_data_dir()
START_DATE = '2015-01-01'
END_DATE = '2026-01-01'
N_SHARDS = 100

UN_POP_MALE_PATH = (
    f'{public_dir}/WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx'
)
UN_POP_FEMALE_PATH = (
    f'{public_dir}/WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx'
)


# 9f4397029d8039e43903bb5b8b7e75ec: (target_country='Japan',
# min_days_per_user=1, min_age=20, max_age=104, min_steps=100, max_steps=50000,
# start_month=11, stop_month=11, exclude_days_of_week=[6],
# exclude_outliers=False, exclude_extreme_values=False).
step_grid = strat.StratifySteps.from_shards(
    path_base=internal_dir,
    start_date=START_DATE,
    end_date=END_DATE,
    n_shards=N_SHARDS,
    un_pop_male_path=UN_POP_MALE_PATH,
    un_pop_female_path=UN_POP_FEMALE_PATH,
    min_days_per_user=1,
    min_age=20,
    max_age=104,
    min_steps=100,
    max_steps=50000,
    target_country='Japan',
    start_month=11,
    stop_month=11,
    exclude_days_of_week=[6],
    override_hash='9f4397029d8039e43903bb5b8b7e75ec',
)

# 2. Load Japan NHNS-J processed dataset (2015-2023) from public_dir
japan_step_path = f'{public_dir}/2015_2024_japan_steps.pkl'
if strat.file_exists(japan_step_path):
  japan_steps_raw = strat.load_pickle(japan_step_path)
  print(f'Loaded NHNS-J step data: {len(japan_steps_raw)} rows.')
else:
  raise FileNotFoundError('NHNS-J processed dataset not found.')

# 3. Load UK participant-level Wave 10 dataset (June 2021 - March 2023) from internal_dir
uk_path_wave10 = f'{internal_dir}/2021_2023_uk_50plus_participant_steps.pkl'

if strat.file_exists(uk_path_wave10):
  uk_participant_df = strat.load_pickle(uk_path_wave10)
  print(f'Loaded UK Wave 10 dataset: {len(uk_participant_df):,} rows.')
else:
  raise FileNotFoundError('UK participant dataset checkpoint not found.')

print('Data loading complete.')

We merge together UN population counts and Fitbit/NHNS-J step data.

In [ ]:
# Free-living linear calibration constants mapping wrist-worn Fitbit step counts
# to waist-worn Yamax pedometer equivalents (Fitbit = 4152.0 + 0.8 * Yamax),
# from Nakagata et al. (2022), Gait & Posture, 98, 24-33,
# https://doi.org/10.1016/j.gaitpost.2022.08.004.
NAKAGATA_INTERCEPT = 4152.0
NAKAGATA_SLOPE = 0.8


def assign_midage_10(age: int) -> int:
  """Maps individual age to 10-year midpoint bracket representing survey strata.

  Args:
      age: Integer age of participant.

  Returns:
      Midpoint integer (25 for 20-29, 35 for 30-39, ..., 75 for 70+).
  """
  if age < 70:
    return age // 10 * 10 + 5
  return 75  # 75 represents 70+ bracket


# 1. Process Fitbit step_grid for Japan
fb_df = step_grid.results_grid.reset_index()
fb_df = fb_df[fb_df['country'] == 'Japan'].copy()
fb_df['midage_10'] = fb_df['midage'].apply(assign_midage_10)

fb_df = (
    fb_df.groupby(['year', 'gender', 'midage_10'])[
        ['steps', 'steps_sq', 'readings']
    ]
    .sum()
    .reset_index()
)
fb_df['gender'] = fb_df['gender'].replace({0: 'male', 1: 'female'})

fb_step_mean = fb_df.copy()
fb_step_mean['mean_steps_fitbit'] = (
    fb_step_mean['steps'] / fb_step_mean['readings']
)
fb_step_mean['var_steps_fitbit'] = (
    fb_step_mean['steps_sq'] / fb_step_mean['readings']
    - fb_step_mean['mean_steps_fitbit'] ** 2
)
fb_step_mean['std_steps_fitbit'] = np.sqrt(fb_step_mean['var_steps_fitbit'])
fb_step_mean = fb_step_mean.rename(columns={'readings': 'n_fitbit'})

# UN population counts for Japan
japan_pop_count = step_grid.un_pop[
    step_grid.un_pop['country'] == 'Japan'
].copy()
japan_pop_count['midage_10'] = japan_pop_count['age'].apply(assign_midage_10)
japan_pop_count['gender'] = japan_pop_count['gender'].replace(
    {0: 'male', 1: 'female'}
)
japan_pop_count = (
    japan_pop_count.groupby(['year', 'gender', 'midage_10'])['Count']
    .sum()
    .reset_index()
)
# UN WPP population counts are reported in thousands
japan_pop_count['n_un'] = japan_pop_count['Count'] * 1000

fb_step_mean = fb_step_mean.merge(
    japan_pop_count[['year', 'gender', 'midage_10', 'n_un']],
    on=['year', 'gender', 'midage_10'],
    how='inner',
)

# 2. Process Japanese NHNS-J steps
japan_steps = japan_steps_raw.copy()
age_dict = {
    '20-29': 25,
    '30-39': 35,
    '40-49': 45,
    '50-59': 55,
    '60-69': 65,
    '70+': 75,
}
japan_steps = japan_steps[japan_steps['age_group'].isin(age_dict.keys())].copy()
japan_steps['midage_10'] = japan_steps['age_group'].map(age_dict)
japan_steps = japan_steps.rename(
    columns={
        'mean_steps': 'mean_steps_nhnsj',
        'std_steps': 'std_steps_nhnsj',
        'n_participants': 'n_nhnsj',
    }
)

# 3. Merge into consolidated japan_estimates dataframe
fb_cols = [
    'year',
    'gender',
    'midage_10',
    'mean_steps_fitbit',
    'std_steps_fitbit',
    'n_fitbit',
    'n_un',
]
japan_cols = [
    'year',
    'gender',
    'midage_10',
    'mean_steps_nhnsj',
    'std_steps_nhnsj',
    'n_nhnsj',
]
japan_estimates = fb_step_mean[fb_cols].merge(
    japan_steps[japan_cols], on=['year', 'gender', 'midage_10'], how='inner'
)

# Filter valid non-COVID study years (2015-2023, excluding 2020-2021)
df_jp_clean = japan_estimates[
    (japan_estimates['gender'] != 'all')
    & (japan_estimates['year'] <= 2023)
    & (~japan_estimates['year'].isin([2020, 2021]))
].copy()

age_label_map = {
    25: '20-29',
    35: '30-39',
    45: '40-49',
    55: '50-59',
    65: '60-69',
    75: '70+',
}
df_jp_clean['age_group'] = df_jp_clean['midage_10'].map(age_label_map)
df_jp_clean['gender_label'] = df_jp_clean['gender'].map(
    {'male': 'Men', 'female': 'Women'}
)
df_jp_clean['mean_steps_fitbit_naka'] = (
    df_jp_clean['mean_steps_fitbit'] - NAKAGATA_INTERCEPT
) / NAKAGATA_SLOPE

# Fixed multi-year UN census population distribution over 2015-2023 (Japan)
jp_stratum_pop = (
    df_jp_clean.groupby(['age_group', 'gender_label'])['n_un']
    .mean()
    .reset_index()
)
jp_total_pop = jp_stratum_pop['n_un'].sum()
jp_stratum_pop['w_un_pop'] = jp_stratum_pop['n_un'] / jp_total_pop
jp_stratum_pop['w_grid'] = 1.0 / 12.0

df_jp_clean = df_jp_clean.merge(
    jp_stratum_pop[['age_group', 'gender_label', 'w_un_pop', 'w_grid']],
    on=['age_group', 'gender_label'],
)

# 4. Clean UK ELSA Participant Dataset
df_uk = uk_participant_df.copy()
if 'midage_uk' in df_uk.columns:
  uk_age_map = {55: '50-59', 65: '60-69', 75: '70-79', 85: '80+'}
  df_uk['age_group'] = df_uk['midage_uk'].map(uk_age_map)

df_uk['age_group'] = pd.Categorical(
    df_uk['age_group'],
    categories=['50-59', '60-69', '70-79', '80+'],
    ordered=True,
)
df_uk['gender'] = pd.Categorical(df_uk['gender'], categories=['male', 'female'])
df_uk_clean = df_uk.dropna(
    subset=['median_daily_steps', 'age_group', 'gender']
).copy()

print(
    'Japan cleaned evaluation rows:'
    f' {len(df_jp_clean)} stratum-year cells (7 years x 12 strata).'
)
print(
    'UK cleaned participant rows:'
    f' {len(df_uk_clean):,} unique participants aged 50+.'
)
print('Data processing and fixed census weights established.')

## Step differences in Japan
We calculate two different measures of discrepancy between the Fitbit and
national estimates: the difference between the overall means between each
cohort, and the average stratum-level difference. The average stratum-level
difference is more informative since it conveys differences in estimates in
specific demographic groups. In order to compute confidence intervals, we
use bootsrap sampling.
In each bootstrap iteration b = 1, ..., 1000, we sample all readings from a year
with replacement. For any point metric theta_hat (MAE or MPD), the 95%
confidence interval is constructed as:
CI_95%(theta) = [theta_hat*_(2.5th percentile),  theta_hat*_(97.5th percentile)]
where theta_hat*_(q) denotes the q-th quantile of the bootstrap samples.

In [ ]:
def evaluate_japan_section1(
    df: pd.DataFrame, n_bootstrap: int = 1000
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
  """Evaluates Difference of Population Means vs Mean of Stratum Differences and formats summary table."""
  np.random.seed(42)
  years = sorted(df['year'].unique())

  states = [
      (
          '1. Raw Unadjusted Fitbit (All Ages 20-70+)',
          df.copy(),
          'mean_steps_fitbit',
      ),
      (
          '2. Nakagata Calibrated (All Ages 20-70+)',
          df.copy(),
          'mean_steps_fitbit_naka',
      ),
      (
          '3. Nakagata Calibrated (Target Ages 20-50 Only)',
          df[df['midage_10'] <= 50].copy(),
          'mean_steps_fitbit_naka',
      ),
  ]

  results = []
  raw_metrics = []

  for label, sub_df, fb_col in states:
    sub = sub_df.copy()
    w_sum = sub.groupby('year')['w_un_pop'].transform('sum')
    sub['w_un_norm'] = sub['w_un_pop'] / w_sum

    # Point estimates
    sub['fb_weighted'] = sub[fb_col] * sub['w_un_norm']
    sub['nh_weighted'] = sub['mean_steps_nhnsj'] * sub['w_un_norm']

    yearly_pop_fb = sub.groupby('year')['fb_weighted'].sum()
    yearly_pop_nh = sub.groupby('year')['nh_weighted'].sum()
    mae_pop_diff = np.mean(np.abs(yearly_pop_fb - yearly_pop_nh))
    mpd_pop_diff = (
        (np.mean(yearly_pop_fb) - np.mean(yearly_pop_nh))
        / np.mean(yearly_pop_nh)
        * 100.0
    )
    mape_pop_diff = np.abs(mpd_pop_diff)

    sub['cell_abs_err'] = np.abs(sub[fb_col] - sub['mean_steps_nhnsj'])
    sub['cell_signed_rel'] = (
        (sub[fb_col] - sub['mean_steps_nhnsj'])
        / sub['mean_steps_nhnsj']
        * 100.0
    )
    sub['cell_abs_rel'] = (
        np.abs(sub[fb_col] - sub['mean_steps_nhnsj'])
        / sub['mean_steps_nhnsj']
        * 100.0
    )
    sub['abs_weighted'] = sub['cell_abs_err'] * sub['w_un_norm']
    sub['signed_rel_weighted'] = sub['cell_signed_rel'] * sub['w_un_norm']
    sub['abs_rel_weighted'] = sub['cell_abs_rel'] * sub['w_un_norm']

    mae_strata_pop = np.mean(sub.groupby('year')['abs_weighted'].sum())
    mpd_strata_pop = np.mean(sub.groupby('year')['signed_rel_weighted'].sum())
    mape_strata_pop = np.mean(sub.groupby('year')['abs_rel_weighted'].sum())
    mae_strata_grid = np.mean(sub['cell_abs_err'])
    mpd_strata_grid = np.mean(sub['cell_signed_rel'])
    mape_strata_grid = np.mean(sub['cell_abs_rel'])

    # Year-Clustered Bootstrap for 95% Confidence Intervals
    boot_metrics = {
        'mae_pop_diff': [],
        'mpd_pop_diff': [],
        'mape_pop_diff': [],
        'mae_strata_pop': [],
        'mpd_strata_pop': [],
        'mape_strata_pop': [],
        'mae_strata_grid': [],
        'mpd_strata_grid': [],
        'mape_strata_grid': [],
    }

    for _ in range(n_bootstrap):
      boot_years = np.random.choice(years, size=len(years), replace=True)
      boot_frames = []
      for rep_id, y in enumerate(boot_years):
        frame = sub[sub['year'] == y].copy()
        frame['rep_id'] = rep_id
        boot_frames.append(frame)
      b_df = pd.concat(boot_frames, ignore_index=True)

      # 1. Population Difference on Bootstrap Sample
      b_pop_fb = b_df.groupby('rep_id')['fb_weighted'].sum()
      b_pop_nh = b_df.groupby('rep_id')['nh_weighted'].sum()
      boot_metrics['mae_pop_diff'].append(np.mean(np.abs(b_pop_fb - b_pop_nh)))
      b_mpd = (
          (np.mean(b_pop_fb) - np.mean(b_pop_nh)) / np.mean(b_pop_nh) * 100.0
      )
      boot_metrics['mpd_pop_diff'].append(b_mpd)
      boot_metrics['mape_pop_diff'].append(np.abs(b_mpd))

      # 2. Stratum Discrepancies on Bootstrap Sample
      boot_metrics['mae_strata_pop'].append(
          np.mean(b_df.groupby('rep_id')['abs_weighted'].sum())
      )
      boot_metrics['mpd_strata_pop'].append(
          np.mean(b_df.groupby('rep_id')['signed_rel_weighted'].sum())
      )
      boot_metrics['mape_strata_pop'].append(
          np.mean(b_df.groupby('rep_id')['abs_rel_weighted'].sum())
      )
      boot_metrics['mae_strata_grid'].append(np.mean(b_df['cell_abs_err']))
      boot_metrics['mpd_strata_grid'].append(np.mean(b_df['cell_signed_rel']))
      boot_metrics['mape_strata_grid'].append(np.mean(b_df['cell_abs_rel']))

    def format_ci_clean(
        val: float, boot_arr: list, is_pct: bool = False
    ) -> str:
      low, high = np.percentile(boot_arr, [2.5, 97.5])
      if is_pct:
        return f'{val:.1f} ({low:.1f}--{high:.1f})'
      else:
        return f'{val:,.1f} ({low:,.1f}--{high:,.1f})'

    raw_metrics.append({
        'label': label,
        'mae_strata_pop': format_ci_clean(
            mae_strata_pop, boot_metrics['mae_strata_pop'], False
        ),
        'mpd_strata_pop': format_ci_clean(
            mpd_strata_pop, boot_metrics['mpd_strata_pop'], True
        ),
        'mape_strata_pop': format_ci_clean(
            mape_strata_pop, boot_metrics['mape_strata_pop'], True
        ),
        'mae_pop_diff': format_ci_clean(
            mae_pop_diff, boot_metrics['mae_pop_diff'], False
        ),
        'mpd_pop_diff': format_ci_clean(
            mpd_pop_diff, boot_metrics['mpd_pop_diff'], True
        ),
        'mape_pop_diff': format_ci_clean(
            mape_pop_diff, boot_metrics['mape_pop_diff'], True
        ),
    })

    def format_ci_detailed(
        val: float, boot_arr: list, is_pct: bool = False
    ) -> str:
      low, high = np.percentile(boot_arr, [2.5, 97.5])
      unit = '%' if is_pct else ' steps/day'
      sign = '+' if is_pct and val > 0 else ''
      return f'{sign}{val:,.1f}{unit} ({sign}{low:,.1f} to {sign}{high:,.1f})'

    results.append({
        'Calibration State & Demographic Scope': label,
        'Difference of Pop. Means (Census-Weighted MAE)': format_ci_detailed(
            mae_pop_diff, boot_metrics['mae_pop_diff']
        ),
        'Difference of Pop. Means (Census-Weighted Signed MPD)': (
            format_ci_detailed(mpd_pop_diff, boot_metrics['mpd_pop_diff'], True)
        ),
        'Difference of Pop. Means (Census-Weighted MAPE)': format_ci_detailed(
            mape_pop_diff, boot_metrics['mape_pop_diff'], True
        ),
        'Mean of Stratum Differences (Census-Weighted MAE)': format_ci_detailed(
            mae_strata_pop, boot_metrics['mae_strata_pop']
        ),
        'Mean of Stratum Differences (Census-Weighted Signed MPD)': (
            format_ci_detailed(
                mpd_strata_pop, boot_metrics['mpd_strata_pop'], True
            )
        ),
        'Mean of Stratum Differences (Census-Weighted MAPE)': (
            format_ci_detailed(
                mape_strata_pop, boot_metrics['mape_strata_pop'], True
            )
        ),
        'Mean of Stratum Differences (Equally-Weighted MAE)': (
            format_ci_detailed(mae_strata_grid, boot_metrics['mae_strata_grid'])
        ),
        'Mean of Stratum Differences (Equally-Weighted Signed MPD)': (
            format_ci_detailed(
                mpd_strata_grid, boot_metrics['mpd_strata_grid'], True
            )
        ),
        'Mean of Stratum Differences (Equally-Weighted MAPE)': (
            format_ci_detailed(
                mape_strata_grid, boot_metrics['mape_strata_grid'], True
            )
        ),
    })

  # Build 5-Column Summary Table (100% Stratum Census-Weighted for All Rows)
  summary_rows = [
      {
          'Age Group': '20–70+',
          'Adjusted': 'No',
          'MAE (steps/day, 95% CI)': raw_metrics[0]['mae_strata_pop'],
          'MPD (%, 95% CI)': raw_metrics[0]['mpd_strata_pop'],
          'MAPE (%, 95% CI)': raw_metrics[0]['mape_strata_pop'],
      },
      {
          'Age Group': '20–70+',
          'Adjusted': 'Yes',
          'MAE (steps/day, 95% CI)': raw_metrics[1]['mae_strata_pop'],
          'MPD (%, 95% CI)': raw_metrics[1]['mpd_strata_pop'],
          'MAPE (%, 95% CI)': raw_metrics[1]['mape_strata_pop'],
      },
      {
          'Age Group': '20–50',
          'Adjusted': 'Yes',
          'MAE (steps/day, 95% CI)': raw_metrics[2]['mae_strata_pop'],
          'MPD (%, 95% CI)': raw_metrics[2]['mpd_strata_pop'],
          'MAPE (%, 95% CI)': raw_metrics[2]['mape_strata_pop'],
      },
  ]
  df_summary_table = pd.DataFrame(summary_rows)

  # Generate Full-Width LaTeX Table Code with explanation in caption
  latex_lines = [
      r'\begin{table*}[htbp]',
      r'\centering',
      (
          r'\caption{Census-weighted stratum-level step count discrepancies and'
          r' relative bias between Fitbit and NHNS-J national pedometer'
          r' benchmarks in Japan (2015--2023). Adjusted estimates were'
          r' calibrated using the free-living calibration equation from'
          r' \citet{Nakagata2022-uo}.}'
      ),
      r'\label{tab:japan_summary_discrepancies}',
      r'\begin{tabular}{lcccc}',
      r'\toprule',
      (
          r'\textbf{Age Group} & \textbf{Adjusted} & \textbf{MAE (steps/day,'
          r' 95\% CI)} & \textbf{MPD (\%, 95\% CI)} & \textbf{MAPE (\%, 95\%'
          r' CI)} \\'
      ),
      r'\midrule',
      (
          f"20--70+ & No & {raw_metrics[0]['mae_strata_pop']} &"
          f" {raw_metrics[0]['mpd_strata_pop']} &"
          f" {raw_metrics[0]['mape_strata_pop']} \\\\"
      ),
      (
          f"20--70+ & Yes & {raw_metrics[1]['mae_strata_pop']} &"
          f" {raw_metrics[1]['mpd_strata_pop']} &"
          f" {raw_metrics[1]['mape_strata_pop']} \\\\"
      ),
      (
          f"20--50 & Yes & {raw_metrics[2]['mae_strata_pop']} &"
          f" {raw_metrics[2]['mpd_strata_pop']} &"
          f" {raw_metrics[2]['mape_strata_pop']} \\\\"
      ),
      r'\bottomrule',
      r'\end{tabular}',
      r'\end{table*}',
  ]
  latex_code = '\n'.join(latex_lines)

  return pd.DataFrame(results), df_summary_table, latex_code


df_section1, df_japan_summary, latex_japan_summary = evaluate_japan_section1(
    df_jp_clean, n_bootstrap=1000
)

print('\n' + '=' * 110)
print(
    'SECTION 1: JAPAN DISCREPANCY SUMMARY TABLE (ALL STRATUM-LEVEL'
    ' CENSUS-WEIGHTED)'
)
print('=' * 110)
display(df_japan_summary)

print('\n' + '=' * 110)
print('FULL-WIDTH LATEX TABLE CODE:')
print('=' * 110)
print(latex_japan_summary)
print('=' * 110)

## Harmonized differences in Japan and England

Published ELSA mean step estimates were obtained by computing marginal means for
age-groups and sex (Brockleband et al., 2026). We try to replicate this on the
Fitbit cohort so that we can quantify the differences between the national
and consumer cohorts.
Per Method 4, builds a symmetrical statistical bridge across Japan and England:

In England Pipeline, we fit the participant level OLS model:
`median_daily_steps ~ C(age_group) + C(gender)`.

In Japan, we fit the stratum level OLS model:
`mean_steps ~ C(age_group) + C(gender_label) + C(year)`.

In [ ]:
# 1. ELSA Harmonized Pipeline
model_uk = smf.ols(
    'median_daily_steps ~ Cat(age_group) + Cat(gender)', data=df_uk_clean
).fit()

uk_age_groups = ['50-59', '60-69', '70-79', '80+']
uk_genders = ['male', 'female']

uk_age_preds = []
for ag in uk_age_groups:
  grid = pd.DataFrame({'age_group': [ag, ag], 'gender': ['male', 'female']})
  pred = model_uk.get_prediction(grid).summary_frame()
  uk_age_preds.append((pred['mean'].mean(), pred['mean_se'].mean()))

uk_gen_preds = []
for gen in uk_genders:
  grid = pd.DataFrame({'age_group': uk_age_groups, 'gender': [gen] * 4})
  pred = model_uk.get_prediction(grid).summary_frame()
  uk_gen_preds.append((pred['mean'].mean(), pred['mean_se'].mean()))

# UK 50+ UN WPP 2024 census population proportions averaged over 2021-2023:
# - Age groups (50-59, 60-69, 70-79, 80+): [0.288, 0.297, 0.261, 0.154]
# - Sex (Male, Female among adults aged 50+): [0.478, 0.522]
uk_age_census_w = [0.288, 0.297, 0.261, 0.154]
uk_gen_census_w = [0.478, 0.522]

# Published ELSA Wave 10 Table 2 Reference Constants (Brocklebank et al., 2026).
# Each tuple has schema:
#   (subgroup, fitbit_mean, fitbit_se, elsa_mean, elsa_ci_low, elsa_ci_high,
#    category, census_weight)
elsa_ref_data = [
    (
        '50-59',
        uk_age_preds[0][0],
        uk_age_preds[0][1],
        9809,
        9463,
        10154,
        'Age',
        uk_age_census_w[0],
    ),
    (
        '60-69',
        uk_age_preds[1][0],
        uk_age_preds[1][1],
        9185,
        8925,
        9446,
        'Age',
        uk_age_census_w[1],
    ),
    (
        '70-79',
        uk_age_preds[2][0],
        uk_age_preds[2][1],
        7413,
        7155,
        7672,
        'Age',
        uk_age_census_w[2],
    ),
    (
        '80+',
        uk_age_preds[3][0],
        uk_age_preds[3][1],
        5294,
        4883,
        5705,
        'Age',
        uk_age_census_w[3],
    ),
    (
        'Men',
        uk_gen_preds[0][0],
        uk_gen_preds[0][1],
        8155,
        7921,
        8389,
        'Sex',
        uk_gen_census_w[0],
    ),
    (
        'Women',
        uk_gen_preds[1][0],
        uk_gen_preds[1][1],
        7696,
        7485,
        7906,
        'Sex',
        uk_gen_census_w[1],
    ),
]

# 2. Japan Harmonized Pipeline
model_jp_fb = smf.ols(
    'mean_steps_fitbit_naka ~ Cat(age_group) + Cat(gender_label) + Cat(year)',
    data=df_jp_clean,
).fit()
model_jp_nh = smf.ols(
    'mean_steps_nhnsj ~ Cat(age_group) + Cat(gender_label) + Cat(year)',
    data=df_jp_clean,
).fit()

jp_age_groups = ['20-29', '30-39', '40-49', '50-59', '60-69', '70+']
jp_genders = ['Men', 'Women']
jp_years = sorted(df_jp_clean['year'].unique())

# Compute Japan UN census demographic weights by age and sex
jp_age_w_dict = df_jp_clean.groupby('age_group')['w_un_pop'].mean()
jp_age_w_sum = jp_age_w_dict.sum()
jp_age_census_w = [jp_age_w_dict[ag] / jp_age_w_sum for ag in jp_age_groups]

jp_gen_w_dict = df_jp_clean.groupby('gender_label')['w_un_pop'].mean()
jp_gen_w_sum = jp_gen_w_dict.sum()
jp_gen_census_w = [jp_gen_w_dict[g] / jp_gen_w_sum for g in jp_genders]

jp_subgroup_data = []
for idx, ag in enumerate(jp_age_groups):
  grid = pd.DataFrame([
      {'age_group': ag, 'gender_label': g, 'year': y}
      for g in jp_genders
      for y in jp_years
  ])
  p_fb = model_jp_fb.get_prediction(grid).summary_frame()
  p_nh = model_jp_nh.get_prediction(grid).summary_frame()
  jp_subgroup_data.append((
      ag,
      p_fb['mean'].mean(),
      p_fb['mean_se'].mean(),
      p_nh['mean'].mean(),
      p_nh['mean_se'].mean(),
      'Age',
      jp_age_census_w[idx],
  ))

for idx, g in enumerate(jp_genders):
  grid = pd.DataFrame([
      {'age_group': ag, 'gender_label': g, 'year': y}
      for ag in jp_age_groups
      for y in jp_years
  ])
  p_fb = model_jp_fb.get_prediction(grid).summary_frame()
  p_nh = model_jp_nh.get_prediction(grid).summary_frame()
  jp_subgroup_data.append((
      g,
      p_fb['mean'].mean(),
      p_fb['mean_se'].mean(),
      p_nh['mean'].mean(),
      p_nh['mean_se'].mean(),
      'Sex',
      jp_gen_census_w[idx],
  ))


def compute_delta_row(
    country: str,
    subgroup: str,
    fb_mean: float,
    fb_se: float,
    ref_mean: float,
    ref_se: float,
) -> Dict[str, Any]:
  """Computes MAE, Signed MPD, MAPE, and 95% Confidence Intervals for an individual subgroup."""
  delta = fb_mean - ref_mean
  mae = np.abs(delta)
  delta_se = np.sqrt(fb_se**2 + ref_se**2)
  mae_low = max(0.0, mae - 1.96 * delta_se)
  mae_high = mae + 1.96 * delta_se

  # Signed MPD (%)
  mpd = (fb_mean / ref_mean - 1.0) * 100.0
  mpd_se = (
      100.0
      * (fb_mean / ref_mean)
      * np.sqrt((fb_se / fb_mean) ** 2 + (ref_se / ref_mean) ** 2)
  )
  mpd_low = mpd - 1.96 * mpd_se
  mpd_high = mpd + 1.96 * mpd_se

  # Absolute MAPE (%) for single subgroup
  mape = np.abs(mpd)
  mape_low = max(0.0, mape - 1.96 * mpd_se)
  mape_high = mape + 1.96 * mpd_se

  return {
      'country': country,
      'subgroup': subgroup,
      'mae_str': f'{mae:,.0f} ({mae_low:,.0f}--{mae_high:,.0f})',
      'mpd_str': f'{mpd:.1f} ({mpd_low:.1f}--{mpd_high:.1f})',
      'mape_str': f'{mape:.1f} ({mape_low:.1f}--{mape_high:.1f})',
  }


def compute_summary_metric_row(
    country: str,
    label: str,
    subgroup_tuples: List[Tuple],
    weight_type: str,
) -> Dict[str, Any]:
  """Computes composite summary MAE, Signed MPD, and MAPE across subgroups with Monte Carlo 95% CIs."""
  np.random.seed(42)
  n_sim = 10000
  n_items = len(subgroup_tuples)

  fb_means = np.array([t[0] for t in subgroup_tuples])
  fb_ses = np.array([t[1] for t in subgroup_tuples])
  ref_means = np.array([t[2] for t in subgroup_tuples])
  ref_ses = np.array([t[3] for t in subgroup_tuples])

  if weight_type == 'equally_weighted':
    weights = np.ones(n_items) / n_items
  else:
    raw_w = np.array([t[4] for t in subgroup_tuples])
    weights = raw_w / np.sum(raw_w)

  # Point estimates of MAE, Signed MPD, and MAPE across subgroups
  abs_errs = np.abs(fb_means - ref_means)
  signed_rel_errs = (fb_means - ref_means) / ref_means * 100.0
  abs_rel_errs = np.abs(fb_means - ref_means) / ref_means * 100.0

  mae_point = np.sum(abs_errs * weights)
  mpd_point = np.sum(signed_rel_errs * weights)
  mape_point = np.sum(abs_rel_errs * weights)

  # Monte Carlo uncertainty simulation
  fb_sims = np.random.normal(fb_means, fb_ses, size=(n_sim, n_items))
  ref_sims = np.random.normal(ref_means, ref_ses, size=(n_sim, n_items))

  abs_sims = np.abs(fb_sims - ref_sims)
  signed_rel_sims = (fb_sims - ref_sims) / ref_sims * 100.0
  abs_rel_sims = np.abs(fb_sims - ref_sims) / ref_sims * 100.0

  mae_sims = np.sum(abs_sims * weights, axis=1)
  mpd_sims = np.sum(signed_rel_sims * weights, axis=1)
  mape_sims = np.sum(abs_rel_sims * weights, axis=1)

  mae_low, mae_high = np.percentile(mae_sims, [2.5, 97.5])
  mpd_low, mpd_high = np.percentile(mpd_sims, [2.5, 97.5])
  mape_low, mape_high = np.percentile(mape_sims, [2.5, 97.5])

  return {
      'country': country,
      'subgroup': label,
      'mae_str': f'{mae_point:,.0f} ({mae_low:,.0f}--{mae_high:,.0f})',
      'mpd_str': f'{mpd_point:.1f} ({mpd_low:.1f}--{mpd_high:.1f})',
      'mape_str': f'{mape_point:.1f} ({mape_low:.1f}--{mape_high:.1f})',
  }


# 1. Process Japan Rows
jp_age_rows = []
jp_gen_rows = []
jp_tuples_age = []
jp_tuples_gen = []
for sub, fb_m, fb_se, nh_m, nh_se, cat, w in jp_subgroup_data:
  row_data = compute_delta_row('Japan', sub, fb_m, fb_se, nh_m, nh_se)
  item = (fb_m, fb_se, nh_m, nh_se, w)
  if cat == 'Age':
    jp_age_rows.append(row_data)
    jp_tuples_age.append(item)
  else:
    jp_gen_rows.append(row_data)
    jp_tuples_gen.append(item)

jp_age_avg = compute_summary_metric_row(
    'Japan', 'Average', jp_tuples_age, 'census_weighted'
)
jp_gen_avg = compute_summary_metric_row(
    'Japan', 'Average', jp_tuples_gen, 'census_weighted'
)

# 2. Process England Rows
uk_age_rows = []
uk_gen_rows = []
uk_tuples_age = []
uk_tuples_gen = []
for sub, fb_m, fb_se, elsa_m, ci_low, ci_high, cat, w in elsa_ref_data:
  elsa_se = (ci_high - ci_low) / (2.0 * 1.96)
  row_data = compute_delta_row('England', sub, fb_m, fb_se, elsa_m, elsa_se)
  item = (fb_m, fb_se, elsa_m, elsa_se, w)
  if cat == 'Age':
    uk_age_rows.append(row_data)
    uk_tuples_age.append(item)
  else:
    uk_gen_rows.append(row_data)
    uk_tuples_gen.append(item)

uk_age_avg = compute_summary_metric_row(
    'England', 'Average', uk_tuples_age, 'census_weighted'
)
uk_gen_avg = compute_summary_metric_row(
    'England', 'Average', uk_tuples_gen, 'census_weighted'
)


# 2.5 Build Intermediate DataFrames for Unified Plotting and Comparisons
elsa_rows = []
elsa_midages = [55, 65, 75, 85, 0, 1]
for idx, (subgroup, fb_m, fb_se, elsa_m, ci_low, ci_high, cat, w) in enumerate(
    elsa_ref_data
):
  elsa_se = (ci_high - ci_low) / (2.0 * 1.96)
  mpd = (fb_m / elsa_m - 1.0) * 100.0
  mpd_se = (
      100.0
      * (fb_m / elsa_m)
      * np.sqrt((fb_se / fb_m) ** 2 + (elsa_se / elsa_m) ** 2)
  )
  elsa_rows.append({
      'category_type': cat,
      'subgroup': subgroup,
      'midage': elsa_midages[idx],
      'fitbit_mean': fb_m,
      'fitbit_se': fb_se,
      'elsa_ref_mean': elsa_m,
      'elsa_ref_se': elsa_se,
      'rel_diff_pct': mpd,
      'rel_diff_se_pct': mpd_se,
      'rel_diff_ci_low_pct': mpd - 1.96 * mpd_se,
      'rel_diff_ci_high_pct': mpd + 1.96 * mpd_se,
  })
elsa_table2_data = pd.DataFrame(elsa_rows)

age_mid_map = {
    '20-29': 25,
    '30-39': 35,
    '40-49': 45,
    '50-59': 55,
    '60-69': 65,
    '70+': 75,
}
jp_summary_rows = []
for sub, fb_m, fb_se, nh_m, nh_se, cat, w in jp_subgroup_data:
  if cat == 'Age':
    mpd = (fb_m / nh_m - 1.0) * 100.0
    mpd_se = (
        100.0
        * (fb_m / nh_m)
        * np.sqrt((fb_se / fb_m) ** 2 + (nh_se / nh_m) ** 2)
    )
    jp_summary_rows.append({
        'age_group': sub,
        'midage': age_mid_map[sub],
        'fb_mean': fb_m,
        'fb_se': fb_se,
        'nh_mean': nh_m,
        'nh_se': nh_se,
        'rel_diff_pct': mpd,
        'rel_diff_se': mpd_se,
        'ci_low': mpd - 1.96 * mpd_se,
        'ci_high': mpd + 1.96 * mpd_se,
    })
japan_summary_df = pd.DataFrame(jp_summary_rows)

jp_sex_rows = []
for sub, fb_m, fb_se, nh_m, nh_se, cat, w in jp_subgroup_data:
  if cat == 'Sex':
    mpd = (fb_m / nh_m - 1.0) * 100.0
    mpd_se = (
        100.0
        * (fb_m / nh_m)
        * np.sqrt((fb_se / fb_m) ** 2 + (nh_se / nh_m) ** 2)
    )
    jp_sex_rows.append({
        'gender_label': sub,
        'fb_mean': fb_m,
        'fb_se': fb_se,
        'nh_mean': nh_m,
        'nh_se': nh_se,
        'rel_diff_pct': mpd,
        'rel_diff_se': mpd_se,
        'ci_low': mpd - 1.96 * mpd_se,
        'ci_high': mpd + 1.96 * mpd_se,
    })
japan_gender_df = pd.DataFrame(jp_sex_rows)


# 3. Build Unified Master Supporting Table

master_table_records = []

# Section 1: Sex Marginals
for r in jp_gen_rows:
  master_table_records.append({
      'Panel': 'Sex Marginals',
      'Country': 'Japan',
      'Subgroup': r['subgroup'],
      'MAE (steps/day, 95% CI)': r['mae_str'],
      'MPD (%, 95% CI)': r['mpd_str'],
      'MAPE (%, 95% CI)': r['mape_str'],
  })
master_table_records.append({
    'Panel': 'Sex Marginals',
    'Country': 'Japan',
    'Subgroup': 'Average',
    'MAE (steps/day, 95% CI)': jp_gen_avg['mae_str'],
    'MPD (%, 95% CI)': jp_gen_avg['mpd_str'],
    'MAPE (%, 95% CI)': jp_gen_avg['mape_str'],
})

for r in uk_gen_rows:
  master_table_records.append({
      'Panel': 'Sex Marginals',
      'Country': 'England',
      'Subgroup': r['subgroup'],
      'MAE (steps/day, 95% CI)': r['mae_str'],
      'MPD (%, 95% CI)': r['mpd_str'],
      'MAPE (%, 95% CI)': r['mape_str'],
  })
master_table_records.append({
    'Panel': 'Sex Marginals',
    'Country': 'England',
    'Subgroup': 'Average',
    'MAE (steps/day, 95% CI)': uk_gen_avg['mae_str'],
    'MPD (%, 95% CI)': uk_gen_avg['mpd_str'],
    'MAPE (%, 95% CI)': uk_gen_avg['mape_str'],
})

# Section 2: Age Marginals
for r in jp_age_rows:
  master_table_records.append({
      'Panel': 'Age Marginals',
      'Country': 'Japan',
      'Subgroup': r['subgroup'],
      'MAE (steps/day, 95% CI)': r['mae_str'],
      'MPD (%, 95% CI)': r['mpd_str'],
      'MAPE (%, 95% CI)': r['mape_str'],
  })
master_table_records.append({
    'Panel': 'Age Marginals',
    'Country': 'Japan',
    'Subgroup': 'Average',
    'MAE (steps/day, 95% CI)': jp_age_avg['mae_str'],
    'MPD (%, 95% CI)': jp_age_avg['mpd_str'],
    'MAPE (%, 95% CI)': jp_age_avg['mape_str'],
})

for r in uk_age_rows:
  master_table_records.append({
      'Panel': 'Age Marginals',
      'Country': 'England',
      'Subgroup': r['subgroup'],
      'MAE (steps/day, 95% CI)': r['mae_str'],
      'MPD (%, 95% CI)': r['mpd_str'],
      'MAPE (%, 95% CI)': r['mape_str'],
  })
master_table_records.append({
    'Panel': 'Age Marginals',
    'Country': 'England',
    'Subgroup': 'Average',
    'MAE (steps/day, 95% CI)': uk_age_avg['mae_str'],
    'MPD (%, 95% CI)': uk_age_avg['mpd_str'],
    'MAPE (%, 95% CI)': uk_age_avg['mape_str'],
})

df_marginal_comparison = pd.DataFrame(master_table_records)

# 4. Generate Concise LaTeX Table Code
latex_lines = [
    r'\begin{table*}[htbp]',
    r'\centering',
    (
        r'\caption{Marginal step count discrepancies and relative bias between'
        r' Fitbit and national benchmarks in Japan (NHNS-J 2015--2023) and'
        r' England (ELSA Wave 10, 2021--2023). Japan estimates are calibrated'
        r' using \citet{Nakagata2022-uo}; England estimates compare commercial'
        r' wrist-worn Fitbit devices against research wrist-worn accelerometers'
        r' without adjustment.}'
    ),
    r'\label{tab:marginal_differences_japan_uk}',
    r'\begin{tabular}{llccc}',
    r'\toprule',
    (
        r'\textbf{Country} & \textbf{Subgroup} & \textbf{MAE (steps/day, 95\%'
        r' CI)} & \textbf{MPD (\%, 95\% CI)} & \textbf{MAPE (\%, 95\% CI)} \\'
    ),
    r'\midrule',
    r'\multicolumn{5}{l}{\textbf{Sex Marginals}} \\',
    r'\midrule',
]
# Sex rows
for _, r in df_marginal_comparison[
    df_marginal_comparison['Panel'] == 'Sex Marginals'
].iterrows():
  sub_clean = r['Subgroup'].replace('–', '--')
  country_clean = r['Country'].replace('–', '--')
  mae_val = r['MAE (steps/day, 95% CI)'].replace('–', '--')
  mpd_val = r['MPD (%, 95% CI)'].replace('–', '--')
  mape_val = r['MAPE (%, 95% CI)'].replace('–', '--')
  if r['Subgroup'] == 'Average':
    latex_lines.append(
        f'\\textit{{{country_clean}}} & \\textit{{{sub_clean}}} &'
        f' \\textit{{{mae_val}}} & \\textit{{{mpd_val}}} &'
        f' \\textit{{{mape_val}}} \\\\'
    )
  else:
    latex_lines.append(
        f'{country_clean} & {sub_clean} & {mae_val} & {mpd_val} & {mape_val}'
        ' \\\\'
    )

latex_lines.extend([
    r'\midrule',
    r'\multicolumn{5}{l}{\textbf{Age Marginals}} \\',
    r'\midrule',
])
# Age rows
for _, r in df_marginal_comparison[
    df_marginal_comparison['Panel'] == 'Age Marginals'
].iterrows():
  sub_clean = r['Subgroup'].replace('–', '--')
  country_clean = r['Country'].replace('–', '--')
  mae_val = r['MAE (steps/day, 95% CI)'].replace('–', '--')
  mpd_val = r['MPD (%, 95% CI)'].replace('–', '--')
  mape_val = r['MAPE (%, 95% CI)'].replace('–', '--')
  if r['Subgroup'] == 'Average':
    latex_lines.append(
        f'\\textit{{{country_clean}}} & \\textit{{{sub_clean}}} &'
        f' \\textit{{{mae_val}}} & \\textit{{{mpd_val}}} &'
        f' \\textit{{{mape_val}}} \\\\'
    )
  else:
    latex_lines.append(
        f'{country_clean} & {sub_clean} & {mae_val} & {mpd_val} & {mape_val}'
        ' \\\\'
    )

latex_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table*}',
])
latex_marginal_comparison = '\n'.join(latex_lines)

print('\n' + '=' * 125)
print('SECTION 3: CONCISE MASTER MARGINAL COMPARISON TABLE (JAPAN & ENGLAND)')
print('=' * 125)
display(
    df_marginal_comparison[[
        'Panel',
        'Country',
        'Subgroup',
        'MAE (steps/day, 95% CI)',
        'MPD (%, 95% CI)',
        'MAPE (%, 95% CI)',
    ]]
)

print('\n' + '=' * 125)
print('LATEX TABLE CODE FOR DIRECT MANUSCRIPT INCLUSION:')
print('=' * 125)
print(latex_marginal_comparison)
print('=' * 125)

In [ ]:
# Figure: Demographic-Dependent Discrepancies (Japan vs. England)
japan_color = '#2b5c8f'  # Steel blue from Japan calibrated analysis
elsa_color = '#e06666'  # Pastel red from England ELSA analysis

elsa_age_summary = (
    elsa_table2_data[elsa_table2_data['category_type'] == 'Age']
    .copy()
    .reset_index(drop=True)
)

elsa_gender_summary = (
    elsa_table2_data[elsa_table2_data['category_type'] == 'Sex']
    .copy()
    .reset_index(drop=True)
)

fig, axs = plt.subplots(
    1, 2, figsize=(7.0, 3.4), sharey=True, gridspec_kw={'width_ratios': [2, 1]}
)

y_ticks_custom = [-10, 0, 10, 20, 30]
y_min_custom = -14
y_max_custom = 34

all_lows = np.concatenate([
    japan_summary_df['ci_low'].values,
    japan_gender_df['ci_low'].values,
    elsa_age_summary['rel_diff_ci_low_pct'].values,
    elsa_gender_summary['rel_diff_ci_low_pct'].values,
])
all_highs = np.concatenate([
    japan_summary_df['ci_high'].values,
    japan_gender_df['ci_high'].values,
    elsa_age_summary['rel_diff_ci_high_pct'].values,
    elsa_gender_summary['rel_diff_ci_high_pct'].values,
])
overall_min_val = np.min(all_lows)
overall_max_val = np.max(all_highs)

# Panel A: Age Groups
ax0 = axs[0]
ax0.set_ylim(y_min_custom, y_max_custom)
ax0.set_yticks(y_ticks_custom)
ax0.set_xlim(20, 93)

ax0.grid(False)
ax0.spines['top'].set_visible(False)
ax0.spines['right'].set_visible(False)
ax0.spines['bottom'].set_linewidth(0.8)
ax0.spines['left'].set_visible(True)
ax0.spines['left'].set_linewidth(0.8)
ax0.spines['left'].set_bounds(overall_min_val, overall_max_val)
ax0.tick_params(axis='both', length=2.5, width=0.6)

ax0.axhline(0, color='#999999', linestyle=':', linewidth=0.8, alpha=0.6)

y_err_low_jp_age = (
    japan_summary_df['rel_diff_pct'] - japan_summary_df['ci_low']
).values
y_err_high_jp_age = (
    japan_summary_df['ci_high'] - japan_summary_df['rel_diff_pct']
).values

y_err_low_el_age = (
    elsa_age_summary['rel_diff_pct'] - elsa_age_summary['rel_diff_ci_low_pct']
).values
y_err_high_el_age = (
    elsa_age_summary['rel_diff_ci_high_pct'] - elsa_age_summary['rel_diff_pct']
).values

age_offset = 0.8

ax0.errorbar(
    japan_summary_df['midage'] - age_offset,
    japan_summary_df['rel_diff_pct'],
    yerr=[y_err_low_jp_age, y_err_high_jp_age],
    fmt='o',
    color=japan_color,
    ecolor=japan_color,
    capsize=3,
    capthick=1.0,
    linewidth=1.2,
    markersize=5,
    label='Japan (NHNS-J, Calibrated)',
)

ax0.errorbar(
    elsa_age_summary['midage'] + age_offset,
    elsa_age_summary['rel_diff_pct'],
    yerr=[y_err_low_el_age, y_err_high_el_age],
    fmt='s',
    color=elsa_color,
    ecolor=elsa_color,
    capsize=3,
    capthick=1.0,
    linewidth=1.2,
    markersize=5,
    label='England (ELSA Wave 10)',
)

for idx, row in japan_summary_df.iterrows():
  val_str = f"{row['rel_diff_pct']:+.1f}"
  ax0.text(
      row['midage'] - age_offset + 1.2,
      row['rel_diff_pct'],
      val_str,
      color=japan_color,
      va='center',
      ha='left',
      fontsize=8,
  )

for idx, row in elsa_age_summary.iterrows():
  val_str = f"{row['rel_diff_pct']:+.1f}"
  ax0.text(
      row['midage'] + age_offset + 1.2,
      row['rel_diff_pct'],
      val_str,
      color=elsa_color,
      va='center',
      ha='left',
      fontsize=8,
  )

ax0.set_xticks([25, 35, 45, 55, 65, 75, 85])
ax0.set_xticklabels(
    ['20-29', '30-39', '40-49', '50-59', '60-69', '70-79\n(70+ Japan)', '80+']
)
ax0.set_title('Age group')
ax0.set_ylabel(
    'Relative\ndifference\n(% of national\nreference)',
    rotation=0,
    ha='right',
    va='center',
)
ax0.legend(loc='upper left', frameon=False)

# Panel B: Sex
ax1 = axs[1]
ax1.set_ylim(y_min_custom, y_max_custom)
ax1.set_yticks(y_ticks_custom)

ax1.grid(False)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_visible(False)
ax1.spines['bottom'].set_linewidth(0.8)
ax1.tick_params(axis='y', left=False, labelleft=False)
ax1.tick_params(axis='x', length=2.5, width=0.6)

ax1.axhline(0, color='#999999', linestyle=':', linewidth=0.8, alpha=0.6)

x_gen_pos = np.array([0, 1])

y_err_low_jp_gen = (
    japan_gender_df['rel_diff_pct'] - japan_gender_df['ci_low']
).values
y_err_high_jp_gen = (
    japan_gender_df['ci_high'] - japan_gender_df['rel_diff_pct']
).values

y_err_low_el_gen = (
    elsa_gender_summary['rel_diff_pct']
    - elsa_gender_summary['rel_diff_ci_low_pct']
).values
y_err_high_el_gen = (
    elsa_gender_summary['rel_diff_ci_high_pct']
    - elsa_gender_summary['rel_diff_pct']
).values

offset = 0.08
ax1.errorbar(
    x_gen_pos - offset,
    japan_gender_df['rel_diff_pct'],
    yerr=[y_err_low_jp_gen, y_err_high_jp_gen],
    fmt='o',
    color=japan_color,
    ecolor=japan_color,
    capsize=3,
    capthick=1.0,
    linewidth=1.2,
    markersize=5,
    label='Japan (NHNS-J, Calibrated)',
)

ax1.errorbar(
    x_gen_pos + offset,
    elsa_gender_summary['rel_diff_pct'],
    yerr=[y_err_low_el_gen, y_err_high_el_gen],
    fmt='s',
    color=elsa_color,
    ecolor=elsa_color,
    capsize=3,
    capthick=1.0,
    linewidth=1.2,
    markersize=5,
    label='England (ELSA Wave 10)',
)

for idx, row in japan_gender_df.iterrows():
  val_str = f"{row['rel_diff_pct']:+.1f}"
  ax1.text(
      x_gen_pos[idx] - offset + 0.1,
      row['rel_diff_pct'],
      val_str,
      color=japan_color,
      va='center',
      ha='left',
      fontsize=8,
  )

for idx, row in elsa_gender_summary.iterrows():
  val_str = f"{row['rel_diff_pct']:+.1f}"
  ax1.text(
      x_gen_pos[idx] + offset + 0.1,
      row['rel_diff_pct'],
      val_str,
      color=elsa_color,
      va='center',
      ha='left',
      fontsize=8,
  )

ax1.set_xticks(x_gen_pos)
ax1.set_xticklabels(['Men', 'Women'])
ax1.set_title('Sex')
ax1.set_xlim(-0.4, 1.6)

plt.suptitle(
    'Demographic-dependent discrepancies between Fitbit\nand national'
    ' surveillance steps in Japan and England',
    y=1.05,
)
plt.show()

# *FIN*